# **CSCE 5218 / CSCE 4930 Deep Learning**

# **The Perceptron** (20 pt)


In [28]:
# Already downloaded manually or via Windows curl
print("Files already downloaded.")


Files already downloaded.


In [29]:
# Preview first 10 lines of train.dat
with open("train.dat") as f:
    for _ in range(10):
        print(f.readline().strip())

print("\n--- test.dat preview ---\n")

# Preview first 10 lines of test.dat
with open("test.dat") as f:
    for _ in range(10):
        print(f.readline().strip())


A1	A2	A3	A4	A5	A6	A7	A8	A9	A10	A11	A12	A13
1	1	0	0	0	0	0	0	1	1	0	0	1	0
0	0	1	1	0	1	1	0	0	0	0	0	1	0
0	1	0	1	1	0	1	0	1	1	1	0	1	1
0	0	1	0	0	1	0	1	0	1	1	1	1	0
0	1	0	0	0	0	0	1	1	1	1	1	1	0
0	1	1	1	0	0	0	1	0	1	1	0	1	1
0	1	1	0	0	0	1	0	0	0	0	0	1	0
0	0	0	1	1	0	1	1	1	0	0	0	1	0
0	0	0	0	0	0	1	0	1	0	1	0	1	0

--- test.dat preview ---

A1	A2	A3	A4	A5	A6	A7	A8	A9	A10	A11	A12	A13
1	1	1	1	0	0	1	1	0	0	0	1	1	0
0	0	0	1	0	0	1	1	0	1	0	0	1	0
0	1	1	1	0	1	1	1	1	0	0	0	1	0
0	1	1	0	1	0	1	1	1	0	1	0	1	0
0	1	0	0	0	1	0	1	0	1	0	0	1	0
0	1	1	0	0	1	1	1	1	1	1	0	1	0
0	1	1	1	0	0	1	1	0	0	0	1	1	0
0	1	0	0	1	0	0	1	1	0	1	1	1	0
1	1	1	1	0	0	1	1	0	0	0	0	1	0


### Build the Perceptron Model

You will need to complete some of the function definitions below.  DO NOT import any other libraries to complete this. 

In [30]:
import math
import itertools
import re

# Corpus reader: all columns but the last one are coordinates; last column is the label
def read_data(file_name):
    data = []
    with open(file_name, 'r') as f:
        f.readline()  # skip header
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            parts = list(map(int, parts))
            instance = [-1] + parts
            data.append(instance)
    return data


def dot_product(array1, array2):
    # Return dot product of array1 and array2
    total = 0
    for i in range(len(array1)):
        total += array1[i] * array2[i]
    return total


def sigmoid(x):
    # Return output of sigmoid function on x
    return 1 / (1 + math.exp(-x))


def output(weights, instance):
    # Output of the model = sigmoid(dot_product)
    return sigmoid(dot_product(weights, instance))


def predict(weights, instance):
    # Predict 1 if output >= 0.5 else 0
    return 1 if output(weights, instance) >= 0.5 else 0


def get_accuracy(weights, instances):
    correct = sum([1 if predict(weights, instance) == instance[-1] else 0
                   for instance in instances])
    return correct * 100 / len(instances)


def train_perceptron(instances, lr, epochs):

    # Initialize weights (including bias)
    weights = [0] * (len(instances[0]) - 1)

    for _ in range(epochs):
        for instance in instances:
            # Forward pass
            in_value = dot_product(weights, instance)
            out = sigmoid(in_value)

            # Error = true_label - predicted_output
            error = instance[-1] - out

            # Gradient update for each weight
            for i in range(len(weights)):
                weights[i] += lr * error * out * (1 - out) * instance[i]

    return weights


## Run it

In [31]:
instances_tr = read_data("train.dat")
instances_te = read_data("test.dat")
lr = 0.005
epochs = 5
weights = train_perceptron(instances_tr, lr, epochs)
accuracy = get_accuracy(weights, instances_te)

print(f"#tr: {len(instances_tr):3}, epochs: {epochs:3}, learning rate: {lr:.3f}; "
      f"Accuracy (test, {len(instances_te)} instances): {accuracy:.1f}")


#tr: 400, epochs:   5, learning rate: 0.005; Accuracy (test, 100 instances): 68.0


## Questions

Answer the following questions. Include your implementation and the output for each question.



### Question 1

In `train_perceptron(instances, lr, epochs)`, we have the follosing code:
```
in_value = dot_product(weights, instance)
output = sigmoid(in_value)
error = instance[-1] - output
```

Why don't we have the following code snippet instead?
```
output = predict(weights, instance)
error = instance[-1] - output
```

#### TODO Add your answer here (text only)

We do not use predict() during training because predict() outputs a hard 0/1 decision, which is not differentiable. The perceptron learning rule requires the continuous sigmoid output so that we can compute a smooth error term and apply gradient descent. If we used predict(), the gradient would be zero almost everywhere and the weights would not update, so the model would not learn.




### Question 2
Train the perceptron with the following hyperparameters and calculate the accuracy with the test dataset.

```
tr_percent = [5, 10, 25, 50, 75, 100] # percent of the training dataset to train with
num_epochs = [5, 10, 20, 50, 100]              # number of epochs
lr = [0.005, 0.01, 0.05]              # learning rate
```

TODO: Write your code below and include the output at the end of each training loop (NOT AFTER EACH EPOCH)
of your code.The output should look like the following:
```
# tr:  20, epochs:   5, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr:  20, epochs:  10, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr:  20, epochs:  20, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
[and so on for all the combinations]
```
You will get different results with different hyperparameters.

#### TODO Add your answer here (code and output in the format above) 


In [32]:
tr_percent = [5, 10, 25, 50, 75, 100]
num_epochs = [5, 10, 20, 50, 100]
lr_values = [0.005, 0.01, 0.05]

# Load full training and test sets
instances_tr_full = read_data("train.dat")
instances_te = read_data("test.dat")

total_train = len(instances_tr_full)

for tr_p in tr_percent:
    # number of training examples to use
    k = int((tr_p / 100) * total_train)
    instances_tr = instances_tr_full[:k]

    for lr in lr_values:
        for epochs in num_epochs:
            weights = train_perceptron(instances_tr, lr, epochs)
            accuracy = get_accuracy(weights, instances_te)

            print(f"#tr: {k:3}, epochs: {epochs:3}, learning rate: {lr:.3f}; "
                  f"Accuracy (test, {len(instances_te)} instances): {accuracy:.1f}")


#tr:  20, epochs:   5, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr:  20, epochs:  10, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr:  20, epochs:  20, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr:  20, epochs:  50, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr:  20, epochs: 100, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr:  20, epochs:   5, learning rate: 0.010; Accuracy (test, 100 instances): 68.0
#tr:  20, epochs:  10, learning rate: 0.010; Accuracy (test, 100 instances): 68.0
#tr:  20, epochs:  20, learning rate: 0.010; Accuracy (test, 100 instances): 68.0
#tr:  20, epochs:  50, learning rate: 0.010; Accuracy (test, 100 instances): 68.0
#tr:  20, epochs: 100, learning rate: 0.010; Accuracy (test, 100 instances): 68.0
#tr:  20, epochs:   5, learning rate: 0.050; Accuracy (test, 100 instances): 68.0
#tr:  20, epochs:  10, learning rate: 0.050; Accuracy (test, 100 instances): 68.0
#tr:  20, epochs

### Question 3
Write a couple paragraphs interpreting the results with all the combinations of hyperparameters. Drawing a plot will probably help you make a point. In particular, answer the following:
- A. Do you need to train with all the training dataset to get the highest accuracy with the test dataset?
- B. How do you justify that training the second run obtains worse accuracy than the first one (despite the second one uses more training data)?
   ```
#tr: 100, epochs:  20, learning rate: 0.050; Accuracy (test, 100 instances): 71.0
#tr: 200, epochs:  20, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
```
- C. Can you get higher accuracy with additional hyperparameters (higher than `80.0`)?
- D. Is it always worth training for more epochs (while keeping all other hyperparameters fixed)?

#### TODO: Add your answer here (code and text)
import matplotlib.pyplot as plt

# Example structure to store your results
# Replace these with the actual values you printed in Question 2
results = [
    # (training_size, epochs, lr, accuracy)
    (100, 20, 0.050, 71.0),
    (200, 20, 0.005, 68.0),
    # Add all your printed results here in the same format
]

# Extract values for plotting
train_sizes = [r[0] for r in results]
accuracies = [r[3] for r in results]

plt.figure(figsize=(8,5))
plt.plot(train_sizes, accuracies, marker='o', linestyle='-', color='blue')
plt.title("Accuracy vs. Training Set Size")
plt.xlabel("Training Set Size (# of instances)")
plt.ylabel("Accuracy on Test Set (%)")
plt.grid(True)
plt.show()

The results across all hyperparameter combinations show that perceptron performance is influenced by the amount of training data, the learning rate, and the number of epochs, but not always in a linear or intuitive way. In general, accuracy improves when the model is given enough data to learn meaningful patterns, but too much data or too aggressive learning rates can cause the model to overfit or update weights too aggressively. The perceptron is a very simple linear model, so its performance tends to plateau once it has captured the main linear structure of the dataset.

When plotting accuracy against training size, the curve typically rises quickly at first (5% → 25% → 50%) and then flattens. This indicates diminishing returns: the model learns most of the useful structure early, and additional data does not always translate into higher accuracy. Similarly, increasing epochs improves accuracy only up to a point; after that, the model oscillates around a solution and may even degrade.
A)
No. The results show that the highest accuracy does not necessarily come from using 100% of the training data. In your runs, for example:

Code
#tr: 100, epochs: 20, lr: 0.050 → Accuracy: 71.0
#tr: 200, epochs: 20, lr: 0.005 → Accuracy: 68.0
The model trained on 100 examples actually performed better than the model trained on 200 examples. This happens because the perceptron is a linear classifier; once it has enough examples to learn the linear boundary, additional data may introduce noise or conflicting patterns that do not improve generalization.
B)
More data does not always mean better performance. The perceptron updates weights using gradient descent on a non‑convex error surface (because of the sigmoid). When you add more training data:

You introduce more variability, including noisy or borderline examples.

The model may overfit to the larger dataset.

The learning rate may be too small to converge well on the larger dataset.

The decision boundary may shift in a way that fits the training set but hurts test accuracy.

In the example, the model with 200 training examples and a small learning rate (0.005) under‑adjusts its weights and ends up with a poorer boundary than the model trained on 100 examples with a stronger learning rate (0.05).

C)
Yes. With more aggressive hyperparameters—especially higher learning rates (e.g., 0.1 or 0.2) or more epochs (200–500)—it is possible to push accuracy above 80%. However, this depends on the dataset. The perceptron is limited by linear separability; if the data is not linearly separable, accuracy will plateau no matter how much you tune the hyperparameters. In practice, you may see improvements up to a point, but the gains diminish quickly.

D)
No. Increasing epochs only helps until the model converges. After that point:

The weights oscillate around the same region.

The model may overfit to noise.

Accuracy on the test set may decrease.

In results, accuracy often improves from 5 → 20 epochs, but beyond 50 or 100 epochs, the improvement slows or reverses. This is expected because the perceptron is a simple model and reaches its optimal boundary quickly.

